# REINFORCE + GAT: nucleon clustering (UrQMD-style **partition loss**)

Trains a **GAT** that outputs **pairwise affinity** (one logit per undirected **kNN** edge: “same cluster” / Bernoulli). A full partition is built by **connected components** on edges that are **on**; there is **no fixed K**.

**Reward (REINFORCE)** is computed **after clusterization** from [`cluster_energy.py`](cluster_energy.py): return is **\(-\mathcal{L}_\mathrm{policy}\)** in **MeV** (same scale as `partition_loss_numpy` × `MEV_PER_GEV`). `train_reinforce` in [`ppo_train.py`](ppo_train.py) centers advantages as **\(R_i - \mathrm{mean}_j R_j\)** over the episodes in each update (batch REINFORCE baseline), not the physics cut baseline. The spatial–momentum cut partition is still used for optional **supervised** warm-start (`baseline_edge_targets`) and for logging **\(\mathcal{L}_\mathrm{baseline}\)** vs policy loss.

The training cell below calls **`train_reinforce`** and refreshes **live matplotlib plots on every update** (IPython `display` / `update_display`).

**Baseline** (from [`nucleons (4).ipynb`](nucleons%20(4).ipynb)): connect pairs if \(\|r_i-r_j\| < R_\text{CUT}\) and \(\|p_i-p_j\| < Q_\text{CUT}\) with **R_CUT = 7 fm**, **Q_CUT = 0.12 GeV/c** in the notebook (UrQMD-style momenta); the pickle stores momenta in **MeV**, so we use **120 MeV/c** for the momentum gate.

**Install** (once):
```bash
pip install torch torch-geometric torch-cluster masstable pylorentz tqdm matplotlib numpy
```

`kNN` edges use **PyG** `torch_geometric.nn.knn_graph` on **phase space** \((r, k)\): positions \(r\) in **fm** and wavenumbers \(k = p/(\hbar c)\) in **fm\(^{-1}\)** with \(p\) in **MeV/c** and \(\hbar c = 197.327\,\mathrm{MeV\cdot fm}\) (equivalently \(k \approx 5.0677\,p\) if \(p\) is in **GeV/c**). Not position alone (requires **`torch-cluster`** wheels matching your PyTorch build).

Optional: pre-generated events in `clustering/datasets/urqmd_nucleons_1k/dataset.pkl` (from `generate_urqmd_nucleon_dataset.py`) are used when present for training sampling and baseline statistics.


In [23]:
%load_ext autoreload
%autoreload 2

import os
import sys
import tempfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, global_mean_pool, knn_graph
from tqdm.auto import tqdm

from cluster_energy import partition_loss_numpy

# All training tensors stay on CPU (default torch device).


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Event stream: UrQMD (optional) or synthetic

In [24]:
import colapy


def extract_nucleons_numpy(particles: list[Any]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Match ``ppo_train.extract_nucleons_numpy``: ``pos`` is ``(N, 4)`` as ``(t, x, y, z)`` (fm/c, fm)."""
    pos, mom, is_proton = [], [], []
    for p in particles:
        if p.pdg_code == 2212:
            mom.append([p.momentum.e, p.momentum.x, p.momentum.y, p.momentum.z])
            pos.append([p.position.t, p.position.x, p.position.y, p.position.z])
            is_proton.append(True)
        elif p.pdg_code == 2112:
            mom.append([p.momentum.e, p.momentum.x, p.momentum.y, p.momentum.z])
            pos.append([p.position.t, p.position.x, p.position.y, p.position.z])
            is_proton.append(False)
    if not pos:
        return np.zeros((0, 4), np.float64), np.zeros((0, 4), np.float64), np.zeros((0,), bool)
    return np.asarray(pos, np.float64), np.asarray(mom, np.float64), np.asarray(is_proton, bool)


class W(colapy.WriterBase):
    events = []

    def __init__(self, **kwargs):
        self.events.clear()

    def __call__(self, event_data):
        self.events.append(event_data)

def try_make_urqmd_event_generator():
    CONFIG = """
<?xml version="1.0" encoding="UTF-8" ?>
<program>
    <generator name="URQMDGenerator"
        pro="197 79"
        tar="197 79"
        nev="1"
        imp="5."
        elb="100."
        tim="200 200"
        generated_config_file="input_file"/>
    <writer name="PythonWriter" class="W"/>
</program>
"""

    def gen_one():
        with tempfile.NamedTemporaryFile(mode="w", suffix=".xml", delete_on_close=False) as tmp:
            tmp.write(CONFIG)
            tmp.close()
            rm = colapy.RunManager().load_module("COLA-Py").load_module("COLA_UrQMD").load_config(tmp.name)
            rm.run(1)
            if os.path.exists("input_file"):
                os.remove("input_file")
        if not W.events:
            return np.zeros((0, 4)), np.zeros((0, 4)), np.zeros((0,), bool)
        ev = W.events[-1]
        pos, mom, isp = extract_nucleons_numpy(ev.particles)
        return pos, mom, isp

    return gen_one


URQMD_GEN = try_make_urqmd_event_generator()


## kNN graph + pairwise edge affinity (no fixed K)

The policy outputs a **Bernoulli logit per candidate edge** (one per undirected kNN candidate pair). Clustering = connected components of edges with sample = 1. The number of clusters is **not** capped by a separate K hyperparameter (only by graph connectivity and sampling).

kNN edges for both the **GAT** `edge_index` and the **action** edge list come from **PyG** `knn_graph` (``torch-cluster``). kNN is built on **CPU** for compatibility (e.g. MPS training). Message-passing edges use ``torch_geometric.utils.to_undirected`` on the directed kNN graph.


In [25]:
"""Graph env + helpers from ``ppo_train`` (single source of truth)."""
from ppo_train import (
    AffinityGraphConfig,
    AffinityGraphEnv,
    HBARC_MEV_FM,
    R_CUT_FM,
    Q_CUT_MEVC,
    baseline_clusters_numpy,
    cluster_labels_from_edges,
    labels_to_partition,
)


## Baseline clustering

Connect pairs when **‖rᵢ−rⱼ‖ < R_CUT** and **‖pᵢ−pⱼ‖ < Q_CUT** (dataset momenta in MeV), then take **connected components**.

**Sanity check:** total `partition_loss_numpy` after baseline clusterization should be **≤** the loss for the naive **monolith** partition `[[0, 1, \dots, n-1]]` (one cluster with every nucleon). That is the “no sensible grouping” upper bound for this energy model on the same full cloud.


In [26]:
# Baseline from nucleons (4).ipynb
R_CUT_FM = 7.0
Q_CUT_GEVC = 0.12
Q_CUT_MEVC = Q_CUT_GEVC * 1000.0


def baseline_clusters_numpy(
    pos: np.ndarray,
    mom: np.ndarray,
    indices: list[int],
    r_cut_fm: float,
    q_cut_momentum: float,
) -> list[list[int]]:
    n = len(indices)
    p3 = mom[:, 1:4]
    adj = [[] for _ in range(n)]
    for a in range(n):
        ia = indices[a]
        for b in range(a + 1, n):
            ib = indices[b]
            if float(np.linalg.norm(pos[ia] - pos[ib])) < r_cut_fm and float(
                np.linalg.norm(p3[ia] - p3[ib])
            ) < q_cut_momentum:
                adj[a].append(b)
                adj[b].append(a)
    used = [False] * n
    comps: list[list[int]] = []
    for s in range(n):
        if used[s]:
            continue
        stack = [s]
        used[s] = True
        comp_local: list[int] = []
        while stack:
            v = stack.pop()
            comp_local.append(indices[v])
            for to in adj[v]:
                if not used[to]:
                    used[to] = True
                    stack.append(to)
        comps.append(sorted(comp_local))
    comps.sort(key=len, reverse=True)
    return comps


def load_valid_events_from_pkl(pkl: Path) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    from generate_urqmd_nucleon_dataset import load_dataset_pickle

    bundle = load_dataset_pickle(pkl)
    out: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
    for ev in bundle.get("events", []):
        if ev is None:
            continue
        m = ev["mom"]
        if m.shape[0] >= 2:
            out.append((ev["pos"], ev["mom"], ev["is_proton"]))
    return out



def baseline_labels_and_partition(
    pos: np.ndarray,
    mom: np.ndarray,
    isp: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[list[int]]]:
    """Align lengths; return (pos_n, mom_n, isp_n, baseline_partition)."""
    n0 = min(pos.shape[0], mom.shape[0], isp.shape[0])
    pos_n = np.asarray(pos[:n0], dtype=np.float64)
    mom_n = np.asarray(mom[:n0], dtype=np.float64)
    isp_n = np.asarray(isp[:n0], dtype=bool)
    n = pos_n.shape[0]
    if n < 2:
        return pos_n, mom_n, isp_n, []
    part = baseline_clusters_numpy(pos_n, mom_n, list(range(n)), R_CUT_FM, Q_CUT_MEVC)
    return pos_n, mom_n, isp_n, part


def labels_from_partition(n: int, part: list[list[int]]) -> np.ndarray:
    lab = np.zeros(n, dtype=np.int32)
    for ci, c in enumerate(part):
        for j in c:
            lab[j] = ci
    return lab


def benchmark_baseline_on_events(
    events: list[tuple[np.ndarray, np.ndarray, np.ndarray]],
) -> None:
    l_list: list[float] = []
    for pos, mom, isp in events:
        n = pos.shape[0]
        if n < 2:
            continue
        part = baseline_clusters_numpy(pos, mom, list(range(n)), R_CUT_FM, Q_CUT_MEVC)
        l_list.append(float(partition_loss_numpy(pos, mom, isp, part)))
    if not l_list:
        print("No events with n≥2 in slice.")
        return
    l_mev = 1000.0 * np.array(l_list)
    n_in = len(events)
    print(
        f"Baseline (slice len={n_in}, evaluated n≥2={len(l_list)}): "
        f"partition loss = {l_mev.mean():.1f} ± {l_mev.std():.1f} MeV"
    )


_VALID_EVENTS = load_valid_events_from_pkl(Path("datasets") / "urqmd_nucleons_1k" / "dataset.pkl")


In [27]:
benchmark_baseline_on_events(_VALID_EVENTS[:64])


Baseline (slice len=64, evaluated n≥2=64): partition loss = -165.2 ± 229.2 MeV


## Baseline clustering — quick 3D view

**Foreground:** multi-body clusters are drawn **on top** (higher `zorder`, larger markers, thicker edges, `depthshade=False`). **Singletons** are smaller and drawn first. **^** = proton, **v** = neutron. Multi-body hues follow sorted cluster id via `tab20` (plot matches legend).


In [ ]:
from matplotlib.lines import Line2D


def _tab20_sample(i: int) -> tuple:
    """Distinct ``tab20`` color for multi-body cluster rank ``i`` (cycles every 20)."""
    cmap = plt.get_cmap("tab20")
    n = int(getattr(cmap, "N", 20))
    t = (float(i % n) + 0.5) / float(n)
    return cmap(t)


def plot_baseline_clusterings_preview(
    events: list[tuple[np.ndarray, np.ndarray, np.ndarray]],
    *,
    seed: int = 0,
) -> None:
    if not events:
        print("No events to visualize.")
        return
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(events), size=len(events), replace=False)
    ncols = min(3, len(events))
    nrows = int(np.ceil(len(events) / ncols))
    fig = plt.figure(figsize=(4.6 * ncols, 4.0 * nrows))
    z_back, z_front = 1, 25
    s_single, lw_single = 40, 0.75
    s_multi, lw_multi = 72, 1.45
    edge_w_leg = 1.1
    for j, ii in enumerate(idx):
        pos, mom, isp = events[int(ii)]
        pos_n, mom_n, isp_n, part = baseline_labels_and_partition(pos, mom, isp)
        n = pos_n.shape[0]
        ax = fig.add_subplot(nrows, ncols, j + 1, projection="3d")
        if n < 2 or not part:
            ax.set_title(f"event {int(ii)}: n<2 or empty", fontsize=9)
            continue
        lab = labels_from_partition(n, part)
        cluster_sizes = np.array([len(c) for c in part], dtype=int)
        singleton = cluster_sizes[lab] == 1
        L_gev = float(partition_loss_numpy(pos_n, mom_n, isp_n, part))
        pr = isp_n
        ne = ~isp_n
        multi_ids = sorted(ci for ci, c in enumerate(part) if len(c) > 1)
        ci_to_rgba = {ci: _tab20_sample(rank) for rank, ci in enumerate(multi_ids)}
        # Background: singletons (draw first, smaller).
        for mask, marker in ((pr, "^"), (ne, "v")):
            ms = mask & singleton
            if np.any(ms):
                ax.scatter(
                    pos_n[ms, 0],
                    pos_n[ms, 1],
                    pos_n[ms, 2],
                    facecolors="none",
                    edgecolors="black",
                    marker=marker,
                    s=s_single,
                    linewidths=lw_single,
                    alpha=0.95,
                    zorder=z_back,
                    depthshade=True,
                )
        # Foreground: multi-body clusters (draw last, larger, no depth fade).
        for mask, marker in ((pr, "^"), (ne, "v")):
            mm = mask & ~singleton
            if np.any(mm):
                idx_mm = np.flatnonzero(mm)
                ec = np.array([ci_to_rgba[int(lab[i])] for i in idx_mm], dtype=float)
                ax.scatter(
                    pos_n[mm, 0],
                    pos_n[mm, 1],
                    pos_n[mm, 2],
                    facecolors="none",
                    edgecolors=ec,
                    marker=marker,
                    s=s_multi,
                    linewidths=lw_multi,
                    alpha=1.0,
                    zorder=z_front,
                    depthshade=False,
                )
        ax.set_xlabel("x [fm]")
        ax.set_ylabel("y [fm]")
        ax.set_zlabel("z [fm]")
        n_multi = int(np.sum(cluster_sizes > 1))
        ax.set_title(
            rf"event {int(ii)}: {n_multi} multi-body / {len(part)} components, "
            rf"$\mathcal{{L}}$={1000.0 * L_gev:.0f} MeV",
            fontsize=9,
        )
        handles: list[Line2D] = []
        for ci in multi_ids:
            rgba = ci_to_rgba[ci]
            handles.append(
                Line2D(
                    [0],
                    [0],
                    linestyle="none",
                    marker="o",
                    markersize=8.0,
                    markerfacecolor="none",
                    markeredgecolor=rgba,
                    markeredgewidth=edge_w_leg,
                    label=f"Cluster {ci} (n={len(part[ci])})",
                )
            )
        handles.append(
            Line2D(
                [0],
                [0],
                linestyle="none",
                marker="^",
                markersize=8.0,
                markerfacecolor="none",
                markeredgecolor="0.35",
                markeredgewidth=edge_w_leg,
                label="Proton",
            )
        )
        handles.append(
            Line2D(
                [0],
                [0],
                linestyle="none",
                marker="v",
                markersize=8.0,
                markerfacecolor="none",
                markeredgecolor="0.35",
                markeredgewidth=edge_w_leg,
                label="Neutron",
            )
        )
        ax.legend(
            handles=handles,
            loc="upper left",
            fontsize=7,
            framealpha=0.92,
            title="Legend",
            title_fontsize=8,
        )
    plt.tight_layout()
    plt.show()


plot_baseline_clusterings_preview(_VALID_EVENTS[:3], seed=1)


## GAT + pairwise edge affinity + Bernoulli REINFORCE

## Run training + monitoring plots

Supervised **warm-start** (optional BCE on baseline edge labels) then **`train_reinforce`**. Both stages use `on_update` so **plots refresh every optimizer step / REINFORCE update** in the live figure (same pattern as `plot_pretrain_history`).

In [ ]:
import importlib
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR

import ppo_train as ppo_train_module
from ppo_train import (
    AffinityGraphConfig,
    AffinityGraphEnv,
    GAT_NODE_IN_DIM,
    GATAffinityPolicy,
    MEV_PER_GEV,
    make_event_sampler,
    train_reinforce,
    train_supervised_edges,
)

importlib.reload(ppo_train_module)


_LIVE_PLOT_DISPLAY_IDS: set[str] = set()
RNG = np.random.default_rng(0)
sample_event = make_event_sampler(_VALID_EVENTS, RNG, None)
cfg = AffinityGraphConfig(k_nn=5)
env = AffinityGraphEnv(cfg)

policy = GATAffinityPolicy(
    in_dim=GAT_NODE_IN_DIM,
    hidden=64,
    n_heads=4,
    n_gat_layers=2,
    edge_mlp_depth=2,
)


In [ ]:
from IPython.display import display, update_display


def plot_pretrain_history(
    h: dict[str, list],
    title: str = "Supervised warmstart",
    *,
    refresh_live: bool = False,
    live_display_id: str | None = None,
) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    fig.suptitle(title)

    ax = axes[0]
    if h.get("supervised_bce"):
        ax.plot(h["supervised_bce"], color="C0", label="supervised BCE")
        ax.legend()
    ax.set_xlabel("warmstart step")
    ax.set_ylabel("BCE")
    ax.set_title("Weighted BCE")
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    if h.get("pretrain_partition_loss"):
        ax.plot(
            np.asarray(h["pretrain_partition_loss"], dtype=np.float64) / MEV_PER_GEV,
            color="C1",
            label=r"$\mathcal{L}$ (policy, GeV)",
        )
    if h.get("pretrain_baseline_loss"):
        ax.plot(
            np.asarray(h["pretrain_baseline_loss"], dtype=np.float64) / MEV_PER_GEV,
            color="C3",
            ls="--",
            alpha=0.85,
            label=r"$\mathcal{L}_\mathrm{base}$ (baseline, GeV)",
        )
    if h.get("pretrain_partition_loss") or h.get("pretrain_baseline_loss") or h.get("pretrain_gap"):
        ax.legend(fontsize=8)
    ax.set_xlabel("warmstart step")
    ax.set_ylabel("GeV")
    ax.set_title("Physics: partition loss — policy vs baseline (÷ MEV_PER_GEV)")
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    if h.get("supervised_pos_weight"):
        ax.plot(h["supervised_pos_weight"], color="C6", label="pos weight")
        ax.legend(loc="upper left")
    if h.get("supervised_coef"):
        ax2 = ax.twinx()
        ax2.plot(h["supervised_coef"], color="C2", ls="--", alpha=0.8, label="supervised coef")
        ax2.set_ylabel("coef", color="C2")
    ax.set_xlabel("warmstart step")
    ax.set_ylabel("weight")
    ax.set_title("Class weight / coef")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if refresh_live and live_display_id:
        if live_display_id not in _LIVE_PLOT_DISPLAY_IDS:
            display(fig, display_id=live_display_id)
            _LIVE_PLOT_DISPLAY_IDS.add(live_display_id)
        else:
            update_display(fig, display_id=live_display_id)
        plt.close(fig)
    else:
        plt.show()


pre_opt = optim.Adam(policy.parameters(), lr=3e-3)
pretrain_history = train_supervised_edges(
    policy,
    env,
    sample_event,
    steps=120,
    events_per_step=32,
    optimizer=pre_opt,
    max_grad_norm=0.5,
    weighted_bce=True,
    pos_weight=None,
    pos_weight_power=0.5,
    pos_weight_max=300.0,
    on_update=lambda h: plot_pretrain_history(
        h, refresh_live=True, live_display_id="ppo-pretrain-history-live"
    ),
)

if pretrain_history.get("supervised_bce"):
    print("pretrain supervised_bce first:", pretrain_history["supervised_bce"][0])
    print("pretrain supervised_bce last: ", pretrain_history["supervised_bce"][-1])

if pretrain_history.get("supervised_pos_weight"):
    w = np.asarray(pretrain_history["supervised_pos_weight"], dtype=np.float64)
    print("pretrain pos_weight mean:", float(np.mean(w)))

if pretrain_history.get("pretrain_partition_loss"):
    v = float(pretrain_history["pretrain_partition_loss"][-1])
    print(f"pretrain L_pol last: {v:.1f} MeV ({v / MEV_PER_GEV:.6g} GeV)")
if pretrain_history.get("pretrain_baseline_loss"):
    v = float(pretrain_history["pretrain_baseline_loss"][-1])
    print(f"pretrain L_base last: {v:.1f} MeV ({v / MEV_PER_GEV:.6g} GeV)")
if pretrain_history.get("pretrain_gap"):
    v = float(pretrain_history["pretrain_gap"][-1])
    print(f"pretrain gap last: {v:.1f} MeV ({v / MEV_PER_GEV:.6g} GeV)")


SupWarmstart:   1%|          | 1/120 [00:21<43:01, 21.69s/it, bce=0.247, L_pol=-9.86e-5, L_base=-0.24, gap=0.24]

In [ ]:
def plot_training_history(
    h: dict[str, list],
    title: str = "REINFORCE + GAT (pairwise)",
    *,
    refresh_live: bool = False,
    live_display_id: str | None = None,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(title)
    ax = axes[0, 0]
    if h.get("episode_return"):
        ax.plot(
            np.asarray(h["episode_return"], dtype=np.float64) / MEV_PER_GEV,
            label="mean return",
            color="C0",
        )
    if h.get("return_baseline"):
        ax.plot(
            np.asarray(h["return_baseline"], dtype=np.float64) / MEV_PER_GEV,
            ls="--",
            alpha=0.8,
            color="C7",
            label="batch mean R (adv. baseline)",
        )
    ax.set_xlabel("update")
    ax.set_ylabel("return (÷ MEV_PER_GEV → GeV scale)")
    if h.get("episode_return") or h.get("return_baseline"):
        ax.legend(fontsize=8)
    ax.set_title("Episode return (higher is better)")
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    if h.get("partition_loss"):
        ax.plot(
            np.asarray(h["partition_loss"], dtype=np.float64) / MEV_PER_GEV,
            label=r"$\mathcal{L}$ (policy, GeV)",
            color="C1",
        )
    if h.get("baseline_loss"):
        ax.plot(
            np.asarray(h["baseline_loss"], dtype=np.float64) / MEV_PER_GEV,
            label=r"$\mathcal{L}_\mathrm{base}$ (baseline, GeV)",
            color="C3",
            ls="--",
            alpha=0.85,
        )
    ax.set_xlabel("update")
    ax.set_ylabel("GeV")
    ax.legend(fontsize=8)
    ax.set_title("Physics: partition loss — policy vs baseline (÷ MEV_PER_GEV)")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    if h.get("policy_loss"):
        ax.plot(h["policy_loss"], label="policy surrogate", color="C2")
    ax.set_xlabel("update")
    ax.set_ylabel("policy term")
    ax.set_title("REINFORCE policy surrogate & LR")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    if h.get("lr"):
        ax_lr = ax.twinx()
        ax_lr.plot(h["lr"], color="C6", ls=":", alpha=0.9, label="lr")
        ax_lr.set_ylabel("learning rate", color="C6")
        ax_lr.tick_params(axis="y", labelcolor="C6")

    ax = axes[1, 1]
    if h.get("edge_entropy"):
        ax.plot(h["edge_entropy"], color="C4", label="edge entropy")
    if h.get("edge_entropy"):
        ax.legend(loc="upper left")
    if h.get("n_clusters"):
        ax2 = ax.twinx()
        ax2.plot(h["n_clusters"], color="C5", alpha=0.75, label="mean # clusters")
        ax2.set_ylabel("# clusters", color="C5")
    ax.set_xlabel("update")
    ax.set_ylabel("entropy")
    ax.set_title("Entropy & clusters")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if refresh_live and live_display_id:
        try:
            from IPython.display import display, update_display
        except ImportError:
            plt.show()
        else:
            if live_display_id not in _LIVE_PLOT_DISPLAY_IDS:
                display(fig, display_id=live_display_id)
                _LIVE_PLOT_DISPLAY_IDS.add(live_display_id)
            else:
                update_display(fig, display_id=live_display_id)
            plt.close(fig)
    else:
        plt.show()


N_REINFORCE_UPDATES = 500
train_opt = optim.Adam(policy.parameters(), lr=4e-4)
train_sched = LinearLR(
    train_opt, start_factor=1.0, end_factor=1e-4 / 4e-4, total_iters=N_REINFORCE_UPDATES
)
history = train_reinforce(
    policy,
    env,
    sample_event,
    optimizer=train_opt,
    lr_scheduler=train_sched,
    n_updates=N_REINFORCE_UPDATES,
    episodes_per_update=8,
    ent_coef=0.01,
    max_grad_norm=0.5,
    on_update=lambda h: plot_training_history(
        h,
        title="REINFORCE + GAT (live)",
        refresh_live=True,
        live_display_id="reinforce-train-history-live",
    ),
)

# Static copy in the notebook output (live cell closes the figure each update).
plot_training_history(history, title="REINFORCE + GAT (final)", refresh_live=False)


## Rollout visualization (deterministic policy, one event)

In [ ]:
@torch.no_grad()
def affinity_rollout(
    env: AffinityGraphEnv,
    policy: GATAffinityPolicy,
    pos: np.ndarray,
    mom: np.ndarray,
    isp: np.ndarray,
) -> tuple[np.ndarray, float, int]:
    """Deterministic: edge **on** if sigmoid(logit) > 0.5. Returns (probs, partition loss (MeV), n_clusters)."""
    policy.eval()
    obs = env.reset(pos, mom, isp)
    logits = policy(obs)
    p = torch.sigmoid(logits)
    ne = int(obs.edge_pair_i.shape[0])
    on = (p > 0.5).float()
    loss_g, labs = env.physics_for_edge_mask(on)
    return p.detach().numpy(), float(loss_g), int(len(np.unique(labs)))


pos, mom, isp = sample_event()
probs, final_L, n_c = affinity_rollout(env, policy, pos, mom, isp)
ne = int(env.n_real_edges)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(np.arange(ne), probs, color="steelblue", alpha=0.85)
axes[0].set_xlabel("edge index (kNN pair)")
axes[0].set_ylabel(r"$\sigma$(logit)")
axes[0].set_title("Pairwise affinity (same-cluster prob.)")
axes[0].axhline(0.5, color="k", ls="--", lw=0.8)
axes[0].grid(True, alpha=0.3)

axes[1].text(
    0.1,
    0.55,
    f"partition loss = {final_L:.1f} MeV\n# clusters ≈ {n_c}",
    fontsize=12,
    transform=axes[1].transAxes,
)
axes[1].axis("off")
axes[1].set_title("Partition summary")
plt.suptitle("Greedy threshold σ > 0.5 on edges")
plt.tight_layout()
plt.show()


In [ ]:
from ppo_train import MEV_PER_GEV, baseline_edge_targets

# Reproduce oracle candidate-gap diagnostic (same split/config as terminal check)
assert _VALID_EVENTS, "Expected preloaded events in _VALID_EVENTS"

rng_oracle = np.random.default_rng(123)
perm_oracle = rng_oracle.permutation(len(_VALID_EVENTS))
test_idx_oracle = perm_oracle[800:960]

cfg_oracle = AffinityGraphConfig(k_nn=6)
env_oracle = AffinityGraphEnv(cfg_oracle)

gaps_oracle = []
for i in test_idx_oracle:
    pos, mom, isp = _VALID_EVENTS[int(i)]
    env_oracle.reset(pos, mom, isp)

    tgt = baseline_edge_targets(env_oracle)
    on = (tgt > 0.5).numpy().astype(bool, copy=False)

    g = env_oracle.graph
    labs = cluster_labels_from_edges(
        env_oracle.pos.shape[0], g.edge_pair_i.numpy(), g.edge_pair_j.numpy(), on
    )
    part_oracle = labels_to_partition(labs)
    l_oracle = float(partition_loss_numpy(env_oracle.pos, env_oracle.mom, env_oracle.isp, part_oracle) * MEV_PER_GEV)

    n_ev = int(env_oracle.pos.shape[0])
    part_b = baseline_clusters_numpy(env_oracle.pos, env_oracle.mom, list(range(n_ev)), R_CUT_FM, Q_CUT_MEVC)
    l_base = float(partition_loss_numpy(env_oracle.pos, env_oracle.mom, env_oracle.isp, part_b) * MEV_PER_GEV)

    gaps_oracle.append(l_oracle - l_base)

print("oracle_candidate_gap_mean", float(np.mean(gaps_oracle)), "MeV")
print("oracle_candidate_gap_std", float(np.std(gaps_oracle)), "MeV")
print("oracle_candidate_gap_median", float(np.median(gaps_oracle)), "MeV")
print("n_eval", len(gaps_oracle))
